# Human video vs robot video for a robot world model

Two world models, same code, same compute and the same number of training frames; only the video differs:
- `wm-phone`: human videos filmed with a phone (`batgre/h2r-phone`);
- `wm-robots`: real robot videos (`batgre/h2r-robots-1h30`), the first videos of a seeded shuffle up to the same
  number of frames.

Each gets a controller trained on the same robosuite episodes (Panda, Lift, `batgre/h2r-sim-v3`), then the planner is
evaluated in closed loop. Runs go to Drive (`MyDrive/h2r/outputs`); after a disconnection, run the cells again.

In [ ]:
# colab
!pip install -q uv
%cd /content
![ -d human-to-robot ] || git clone https://github.com/batgr/human-to-robot
%cd /content/human-to-robot
!git pull && uv sync

In [ ]:
# colab
import os
from getpass import getpass

from google.colab import drive

os.environ["HF_TOKEN"] = getpass("HF token: ")
os.environ["WANDB_API_KEY"] = getpass("W&B key: ")
os.environ["MUJOCO_GL"] = "egl"  # offscreen rendering for robosuite
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/h2r/outputs && ln -sfn /content/drive/MyDrive/h2r/outputs outputs

In [ ]:
import json

from IPython.display import Image, display

PHONE = "data.root=data/h2r-phone/ data.hf_repo=batgre/h2r-phone"
ROBOTS = "data.root=data/h2r-robots-1h30/ data.hf_repo=batgre/h2r-robots-1h30 data.max_videos=102"  # 2,587 train frames, phone: 2,580
# world models: ~2,000 train clips each, 1,000 x 128 clips = 60 passes, best checkpoint on validation every 50;
# controllers: 61,700 transitions, 4,000 x 256 = 17 passes (the 2026-10-14 ablation peaked between 3,250 and 4,750)
WM_STEPS, WM_EVAL, CTRL_STEPS, EPISODES = 1000, 50, 4000, 30

## 1. Data

In [ ]:
!uv run hf download batgre/h2r-phone --repo-type dataset --local-dir data/h2r-phone --exclude "raw/*"
!uv run hf download batgre/h2r-robots-1h30 --repo-type dataset --local-dir data/h2r-robots-1h30

## 2. World models

Validation every 50 iterations: skill(h) = 1 - error / error of persistence for h = 1..5, controllability, effective
rank, active latent dims, straightness. `curves.png` in each run has every training and validation curve.

In [ ]:
!uv run h2r train --set name=wm-phone {PHONE} iterations={WM_STEPS} eval_every={WM_EVAL}
!uv run h2r train --set name=wm-robots {ROBOTS} iterations={WM_STEPS} eval_every={WM_EVAL}

In [ ]:
for run in ["wm-phone", "wm-robots"]:
    !grep -E "^step|^done" outputs/{run}/log.txt | tail -2
    display(Image(f"outputs/{run}/curves.png", width=1000))

## 3. Controllers, on the same robosuite episodes

In [ ]:
!uv run hf download batgre/h2r-sim-v3 --repo-type dataset --local-dir data/h2r-sim
for wm in ["phone", "robots"]:
    !uv run h2r controller train --set name=ctrl-{wm} world_model=outputs/wm-{wm} iterations={CTRL_STEPS} eval_every=100

In [ ]:
for run in ["ctrl-phone", "ctrl-robots"]:
    !grep -E "^done" outputs/{run}/log.txt
    display(Image(f"outputs/{run}/curves.png", width=700))

## 4. Pick success in closed loop

random (the floor), replay (the action interface, open loop) and the planner with each world model, on the same test
episodes; Wilson 95% intervals.

In [ ]:
!uv run h2r eval pick --set name=pick-baselines "policies=[random, replay]" episodes={EPISODES}
for wm in ["phone", "robots"]:
    !uv run h2r eval pick --set name=pick-{wm} "policies=[planner]" controller=outputs/ctrl-{wm} episodes={EPISODES}

In [ ]:
for run in ["pick-baselines", "pick-phone", "pick-robots"]:
    for policy, result in json.load(open(f"outputs/{run}/results.json"))["summary"].items():
        print(run, policy, result)
for wm in ["phone", "robots"]:
    for kind in ["success", "failure"]:
        if os.path.exists(f"outputs/pick-{wm}/episode_{kind}.png"):
            display(Image(f"outputs/pick-{wm}/episode_{kind}.png", width=900))

## 5. Publish the trained models (config and best checkpoint of each run)

In [ ]:
for run in ["wm-phone", "wm-robots", "ctrl-phone", "ctrl-robots"]:
    !uv run hf upload batgre/h2r-lift outputs/{run} {run} --include config.yaml --include best.pt --include curves.png